<a href="https://colab.research.google.com/github/CristhianSeverino/Data_Engeerig/blob/main/Learn_SQL_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SQL Practice Notebook**

> Created By: **Cristhian Calle Severino**.

**If you found this Notebook helpful, I'd love to hear about it!**
**hae fun Crreatig!**

*   **Github**: https://github.com/CristhianSeverino
*   **Linkedin**: https://www.linkedin.com/in/cristhianandrescalleseverino/








>This Project **Includes:**😎

* **Synthectic Data Creation:** Five Synthetic tables are Created Using The Faker Library, Adhering to a star schema desing.⭐
* **Database Exploration and Connection:** connection to and exploration of the database using notebook magic commands (or "magic Methods").💽
* **Includes the Correct Answer** and a test/verification.🕵🏽‍♂️


# **Library Instalattion And Import 📚**

In [1]:
!pip install faker
!pip install pandas sqlalchemy mysqlclient faker
!pip install ipython-sql==0.5.0 prettytable==3.9.0

print("="*150)
print(" "*50+"Installed Libraries 📚 ;)")
print("="*150)
from faker import Faker
import pandas as pd
from datetime import datetime
import random
from sqlalchemy import create_engine
import numpy as np
import os as os
import locale



print("="*150)
print(" "*50+"Imported Libraries ;)")
print("="*150)

%load_ext sql
print("="*150)
print(" "*50+"Loaded SQL Extesion ;)")
print("="*150)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 1.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for mysqlclient: filename=mysqlclient-2.2.7-cp312-cp312-linux_x86_64.whl size=129065 sha256=1fee772731311ed4678f099eb761761a12206a913b3c4190c3489f21e32256c5
  Stored in directory: /root/.cache/pip/wheels/27/95/18/7f176fffd46629e710c04c810b9c4d7d4358fe7c96a7d2306d
Successfully built mysqlclient
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.1 MB/s eta 0:00:00
  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.16.0
    Uninstalling prettytable-3.16.0:
      Successfully uninstalled prettytable-3.16.0
                                                  Installed Libraries 📚 ;)
                                                  Imported Libraries ;

# **Sythetic Data Creation 🤖**

  * This othebook and its SQl exercises rely on adatabase generated with synthetic data. In this section, you'll find the functions and funtion execution that make this possible.

  * Adapt them based on what you need or want to create. **Feel free to expand the datasets to your liking! 😎**.

  * The database is loaded in this section into SQLite, which is **ideal for practicting SQL within a Colab enviroment**.
  
**Enjoy exploring and creating with this proyect. Have a productive Day! ☕**


In [2]:
#================================= Generating The Star Schema =============================================
# Modify num_fact_rows as needed
def generate_star_schema(num_fact_rows=50000):
    # Note: 'es_CO' locale is maintained as it dictates the nature of the synthetic data (e.g., names, addresses)
    fake = Faker('es_CO')

    #======================= Dimension Tables ===============

    # 1. Customer Dimension: Simulates customer base data
    dim_customer_data = []
    used_ids = set()
    num_customers = 10000 # Define the Number of Clients
    while len(dim_customer_data) < num_customers: # We use num_customers in the loop
        id_candidate = fake.random_int(min=1000000000, max=1000100000)
        if id_candidate not in used_ids:
            used_ids.add(id_candidate)
            dim_customer_data.append({
                'customer_id': id_candidate,
                'name': fake.name(),
                'category': fake.random_element(['BtoB', 'BtoC']),
                'email': fake.email(),
                'birth_date': fake.date_of_birth(minimum_age=18, maximum_age=45),
                'phone': fake.phone_number(),
            })
    dim_customer = pd.DataFrame(dim_customer_data)

    # 2. Product Dimension: Simulates different products.
    num_products = 350
    dim_product_data = []
    for i in range(num_products):
        dim_product_data.append({
            'product_id': i,
            'product_name': fake.catch_phrase(),
            'category': fake.random_element(['Electronics', 'Apparel', 'Toys']),
            'unit_price': round(random.uniform(5.0, 500.0), 2),
            'stock_code': f"PROD{i+1:04d}"
        })
    dim_product = pd.DataFrame(dim_product_data)

    # 3. Time Dimension: Record the date of the Transactions
    dates = pd.to_datetime(pd.date_range('2024-01-01', '2024-12-31'))
    dim_time_data = []
    for i, date in enumerate(dates):
        dim_time_data.append({
            'date_id': i,
            'date': date,
            'year': date.year,
            'month': date.month,
            'day': date.day,
            'day_of_week': date.day_name(),
        })
    dim_time = pd.DataFrame(dim_time_data)

    # 4. Location Dimension: Breaks down the address
    cities = ['Manizales', 'Bogota', 'Pereira', 'Medellin', 'Cartagena', 'Barranquilla', 'Santa Marta', 'Neiva', 'Mitu', 'Valledupar', 'Mocoa']
    num_locations = len(cities)
    dim_location_data = []
    for i, city_name in enumerate(cities):
        dim_location_data.append({
            'location_id': i,
            'city_name': city_name,
            'address': fake.address(),
        })
    dim_location = pd.DataFrame(dim_location_data)

    # 5. Fact table (Sales)
    fact_sales_data = {
        'sale_id': np.arange(num_fact_rows),
        # Foreign Keys of dimension tables
        'customer_id': np.random.choice(dim_customer['customer_id'].values, num_fact_rows, replace=True),
        'product_id': np.random.randint(0, num_products, num_fact_rows),
        'date_id': np.random.randint(0, len(dates), num_fact_rows),
        'location_id': np.random.randint(0, num_locations, num_fact_rows),
        # Business Metrics
        'quantity_sold': np.random.randint(1, 10, num_fact_rows),
    }
    fact_sales = pd.DataFrame(fact_sales_data)

    # Calculate total revenue based on products prices
    fact_sales = fact_sales.merge(
        dim_product[['product_id', 'unit_price']],
        on='product_id',
        how='left'
    )
    fact_sales['total_revenue'] = fact_sales['unit_price'] * fact_sales['quantity_sold']
    fact_sales = fact_sales.drop(columns=['unit_price'])

    return {
        'fact_sales': fact_sales,
        'dim_customer': dim_customer,
        'dim_product': dim_product,
        'dim_time': dim_time,
        'dim_location': dim_location
    }


In [3]:
#================================= Generate SQL Database =============================================
# Remember, this function heavily relies on the structure created in the previous function.
# Before modifying, ensure it's consistent with the previous function and the execution cell. ;)
def load_to_sqlite(dataframes_dict, db_path='eschema_business.db'):
    """
    Loads the dictionary of pandas DataFrames into an SQLite database.
    The database is saved to a local file.

    Args:
        dataframes_dict (dict): A dictionary containing the DataFrames to be loaded.
        db_path (str): The file path where the SQLite database will be saved.
    """
    try:
        # Connection string to a local SQLite file
        engine_string = f'sqlite:///{db_path}'
        engine = create_engine(engine_string)

        # If the file already exists, we delete it to recreate clean on each run
        if os.path.exists(db_path):
            os.remove(db_path)
            print(f"🔄 Existing Database File '{db_path}' Existing File Deleted.")

        print(f"🔗 Database Connection '{db_path}' Established.")

        # Load each DataFrame into its respective table
        for table_name, df in dataframes_dict.items():
            print(f"⏳ Loading Table: '{table_name}'...")

            # 'if_exists'='replace' creates the table if it doesn't exist or replaces it if it already does
            df.to_sql(
                name=table_name,
                con=engine,
                if_exists='replace',
                index=False,
            )
            print("="*150)
            print(f"{' '} ✅ Table '{table_name}' successfully loaded.")
            print("="*150)

        print("\n🎉 All data has been loaded into SQLite!")

    except Exception as e:
        print(f"❌ An error occurred while loading data: {e}")


**Ejecutar Funciones**

In [4]:
#========================================== Function Execution =======================================

# 1. Generate the DataFrames with the `generate_star_schema` function
print("Starting the generation of business data... 📈")

# You can adjust the number of rows in the fact table here
tables_business = generate_star_schema(num_fact_rows=100000)

print("Data generated successfully. 🎉")

# 2. Load the DataFrames into the SQLite database with `load_to_sqlite`
print("Starting to load data into the SQLite database... 💾")
# The function will create a file called 'business_sales_schema.db'
load_to_sqlite(tables_business)

print("Full charge. ✅")

Starting the generation of business data... 📈
Data generated successfully. 🎉
Starting to load data into the SQLite database... 💾
🔗 Database Connection 'eschema_business.db' Established.
⏳ Loading Table: 'fact_sales'...
  ✅ Table 'fact_sales' successfully loaded.
⏳ Loading Table: 'dim_customer'...
  ✅ Table 'dim_customer' successfully loaded.
⏳ Loading Table: 'dim_product'...
  ✅ Table 'dim_product' successfully loaded.
⏳ Loading Table: 'dim_time'...
  ✅ Table 'dim_time' successfully loaded.
⏳ Loading Table: 'dim_location'...
  ✅ Table 'dim_location' successfully loaded.

🎉 All data has been loaded into SQLite!
Full charge. ✅


# **Ejercicios SQL**

#**SQL Exercises**
*Let's Get Started! (o Dive In!)*

>Here you'll find 10 basic SQL exercises. Feel free to add cells and create more complex queries. **Ask Yourself the Following Questions**:🤔☕🧮



* If I were the **CEO**, what would I want to see? What **Key Performance Indicators (KPIs)** provide value and save me time?

* If I were the **Senior Data Analyst**, what information would I analyze in search of **high-value insights**?

* What indicators should I check **daily, weekly**, and **quarterly**?




In [5]:
%load_ext sql
%sql sqlite:///eschema_business.db
#Load DataBase
print("-"*50)
print("Database Eschema_Business Loaded 💽💽💽")
print("-"*50)

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
--------------------------------------------------
Database Eschema_Business Loaded 💽💽💽
--------------------------------------------------


In [7]:
%sql SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///eschema_business.db
Done.


name
fact_sales
dim_customer
dim_product
dim_time
dim_location


1. **View Only 10 Sales Records**

This query lets you preview ***the first 10 rows*** of the fact_sales table, which contains the core business metrics.



In [8]:
%%sql SELECT * FROM fact_sales LIMIT 10;

 * sqlite:///eschema_business.db
Done.


sale_id,customer_id,product_id,date_id,location_id,quantity_sold,total_revenue
0,1000008468,243,54,9,4,1112.72
1,1000023559,254,132,6,4,339.36
2,1000070617,174,308,5,2,804.12
3,1000018219,259,265,7,8,2094.32
4,1000010071,1,123,5,1,191.71
5,1000090197,103,83,0,8,1360.88
6,1000087969,81,171,8,7,1488.3400000000001
7,1000076637,25,144,6,3,981.81
8,1000055840,39,307,1,8,346.96
9,1000097281,106,61,1,6,381.12



>%%sql
SELECT *
FROM fact_sales
LIMIT 10;



---
2. **Count the Total Number of Sales:** This query lets you determine the total number of transactions recorded in the fact table.


In [9]:
%%sql SELECT COUNT(*) AS total_revenue FROM fact_sales;

 * sqlite:///eschema_business.db
Done.


total_revenue
100000


>%%sql SELECT COUNT(*) AS total_revenue FROM fact_sales;



---
3. **Calculate Total Revenue:** This query sums all the values in the total_revenue column to get the overall revenue generated.


In [10]:
%%sql SELECT SUM(total_revenue) AS net_reveue FROM fact_sales;

 * sqlite:///eschema_business.db
Done.


net_reveue
119702077.65999751





>%%sql SELECT SUM(total_revenue) AS net_reveue FROM fact_sales;




---
4. **Find the Top 5 Sales by Revenue**

    By using ORDER BY and DESC, you can sort the results from highest to lowest to easily find the largest transactions.

In [11]:
%%sql
SELECT *
FROM fact_sales
ORDER BY total_revenue DESC
LIMIT 5;


 * sqlite:///eschema_business.db
Done.


sale_id,customer_id,product_id,date_id,location_id,quantity_sold,total_revenue
2916,1000047971,244,138,10,9,4499.7300000000005
5020,1000015781,244,199,6,9,4499.7300000000005
12550,1000074332,244,24,5,9,4499.7300000000005
15395,1000022093,244,325,10,9,4499.7300000000005
16483,1000096367,244,115,1,9,4499.7300000000005


> %%sql

> SELECT *

> FROM fact_sales

> ORDER BY total_revenue DESC

> LIMIT 5;



---
5. **Contar el número de clientes únicos:**


In [12]:
%%sql
SELECT COUNT(DISTINCT customer_id) AS total_unique_customers
FROM dim_customer;


 * sqlite:///eschema_business.db
Done.


total_unique_customers
10000


> %%sql

> SELECT COUNT(DISTINCT customer_id) AS total_unique_customers

> FROM dim_customer;



---
6. **Sum Quantity Sold by Product Category** By using GROUP BY, you can group the sales by category and sum the quantities sold, joining the fact table with the dimension table.

In [13]:
%%sql

SELECT
    dp.category,
    SUM(fs.quantity_sold) AS total_quantity_sold
FROM
    fact_sales fs
JOIN
    dim_product dp ON fs.product_id = dp.product_id
GROUP BY
    dp.category;

 * sqlite:///eschema_business.db
Done.


category,total_quantity_sold
Apparel,172041
Electronics,145099
Toys,183192


> SELECT

    dp.category,
    SUM(fs.quantity_sold) AS total_quantity_sold

> FROM

    fact_sales fs

> JOIN
    dim_product dp ON fs.product_id = dp.product_id

> GROUP BY
    dp.category;

---
7. **Calculate Average Revenue Per Month** This query joins the fact table with the time dimension table to calculate the average revenue generated for each month.


In [14]:
%%sql
SELECT
    dt.year,
    dt.month,
    AVG(fs.total_revenue) AS average_revenue
FROM
    fact_sales fs
JOIN
    dim_time dt ON fs.date_id = dt.date_id
GROUP BY
    dt.year, dt.month
ORDER BY
    dt.year, dt.month;

 * sqlite:///eschema_business.db
Done.


year,month,average_revenue
2024,1,1206.9222190577868
2024,2,1192.0578547979778
2024,3,1193.2401128586846
2024,4,1194.4860407267429
2024,5,1193.6927922232805
2024,6,1210.9309991512052
2024,7,1212.3460044827234
2024,8,1176.0827835538691
2024,9,1194.224448600381
2024,10,1197.9743105677774


>%%sql
SELECT
    dt.year,
    dt.month,
    AVG(fs.total_revenue) AS average_revenue
FROM
    fact_sales fs
JOIN
    dim_time dt ON fs.date_id = dt.date_id
GROUP BY
    dt.year, dt.month
ORDER BY
    dt.year, dt.month;



---
8. **Find the Top-Selling Products:** This query lets you identify the most popular products based on the total quantity sold.


In [15]:
%%sql
SELECT
    dp.product_id,
    dp.product_name,
    dp.category,
    SUM(fs.quantity_sold) AS total_quantity_sold
FROM
    fact_sales fs
JOIN
    dim_product dp ON fs.product_id = dp.product_id
GROUP BY
    dp.product_id, dp.product_name, dp.category
ORDER BY
    total_quantity_sold DESC
LIMIT 5;

 * sqlite:///eschema_business.db
Done.


product_id,product_name,category,total_quantity_sold
346,Organic static functionalities,Apparel,1720
244,Virtual scalable Internet solution,Apparel,1713
89,Centralized explicit orchestration,Toys,1671
6,Switchable mission-critical structure,Electronics,1659
60,Public-key stable leverage,Toys,1640


>%%sql
SELECT
    dp.product_id,
    dp.product_name,
    dp.category,
    SUM(fs.quantity_sold) AS total_quantity_sold
FROM
    fact_sales fs
JOIN
    dim_product dp ON fs.product_id = dp.product_id
GROUP BY
    dp.product_id, dp.product_name, dp.category
ORDER BY
    total_quantity_sold DESC
LIMIT 5;



---
9. **Filter Sales by a Specific Category** This query uses the WHERE clause to filter all sales that belong to the 'Electronics' category.



In [17]:
%%sql
SELECT
    fs.sale_id,
    fs.customer_id,
    fs.product_id,
    dp.product_name,
    dp.category,
    fs.quantity_sold,
    fs.total_revenue,
    fs.date_id,
    fs.location_id
FROM
    fact_sales fs
JOIN
    dim_product dp ON fs.product_id = dp.product_id
WHERE
    dp.category = 'Electronics'
LIMIT 15;

 * sqlite:///eschema_business.db
Done.


sale_id,customer_id,product_id,product_name,category,quantity_sold,total_revenue,date_id,location_id
322,1000075835,4,Phased dynamic time-frame,Electronics,8,3950.32,278,1
792,1000007579,4,Phased dynamic time-frame,Electronics,1,493.79,95,4
1314,1000001557,4,Phased dynamic time-frame,Electronics,8,3950.32,271,10
1942,1000053515,4,Phased dynamic time-frame,Electronics,7,3456.53,315,1
3521,1000044005,4,Phased dynamic time-frame,Electronics,7,3456.53,130,0
3533,1000080715,4,Phased dynamic time-frame,Electronics,3,1481.3700000000001,92,0
3739,1000011581,4,Phased dynamic time-frame,Electronics,6,2962.7400000000002,53,3
3753,1000091302,4,Phased dynamic time-frame,Electronics,7,3456.53,315,9
4072,1000029714,4,Phased dynamic time-frame,Electronics,2,987.58,141,1
4328,1000058972,4,Phased dynamic time-frame,Electronics,9,4444.110000000001,206,1


>%%sql
SELECT
    fs.sale_id,
    fs.customer_id,
    fs.product_id,
    dp.product_name,
    dp.category,
    fs.quantity_sold,
    fs.total_revenue,
    fs.date_id,
    fs.location_id
FROM
    fact_sales fs
JOIN
    dim_product dp ON fs.product_id = dp.product_id
WHERE
    dp.category = 'Electronics'
LIMIT 15;



---
10. **Count Sales by Day of the Week** This query joins the fact table with the time dimension table to determine which days of the week recorded the most sales.

In [18]:
%%sql
SELECT
    dt.day_of_week,
    COUNT(fs.sale_id) AS total_sales
FROM
    fact_sales fs
JOIN
    dim_time dt ON fs.date_id = dt.date_id
GROUP BY
    dt.day_of_week
ORDER BY
    total_sales DESC;

 * sqlite:///eschema_business.db
Done.


day_of_week,total_sales
Monday,14614
Tuesday,14516
Saturday,14277
Thursday,14207
Sunday,14195
Wednesday,14153
Friday,14038




>%%sql
%%sql
SELECT
    dt.day_of_week,
    COUNT(fs.sale_id) AS total_sales
FROM
    fact_sales fs
JOIN
    dim_time dt ON fs.date_id = dt.date_id
GROUP BY
    dt.day_of_week
ORDER BY
    total_sales DESC;


# **HAVE FUN CREATING🤗**

*If this project has helpful to you, stop by my Linkedin ad let me know how it helped! ☕*

* **linkedIn:** https://www.linkedin.com/in/cristhianandrescalleseverino/

I've left the following cells open for you to add your personalized queries. Go ahead and **Create**.

***Being 1% better every day is the key 🔥***

